# EarLoop — baseline: pairwise preference learning (A/B)

Этот ноутбук — **baseline** для дипломной задачи:
- есть *мини-датасет* профилей (перцептивные параметры + 23-полосная кривая),
- есть A/B пары с выбором пользователя,
- обучаем простую **Bradley–Terry / pairwise logistic regression** модель предпочтений,
- считаем первые метрики.

> Важно: это *синтетический* мини-датасет для отладки пайплайна. В реальном продукте пары приходят онлайн от пользователя.


In [ ]:
import pandas as pd
import numpy as np

profiles = pd.read_csv("./profiles_23band.csv")
pairs = pd.read_csv("./ab_pairs.csv")

feat_cols = ["bass","tilt","presence","air"]
X = profiles[feat_cols].to_numpy()

profiles.head(), pairs.head()


## Baseline-модель

Пусть пользователь имеет скрытый вектор вкуса **w** в перцептивном пространстве.
Для A/B пары вероятность выбрать A:

\[
P(A \succ B) = \sigma( w^T (x_A - x_B) )
\]

где \(\sigma\) — сигмоида. Это классическая модель Bradley–Terry для парных сравнений.


In [ ]:
def sigmoid(z):
    return 1/(1+np.exp(-z))

def train_pairwise_logreg(pairs_sub, lr=0.15, epochs=10, l2=1e-3, seed=42):
    rng = np.random.default_rng(seed)
    w = np.zeros(len(feat_cols), dtype=float)
    for ep in range(epochs):
        perm = rng.permutation(len(pairs_sub))
        for i in perm:
            row = pairs_sub.iloc[i]
            a = int(row.a_id); b = int(row.b_id); y = int(row.y_a_wins)
            d = X[a] - X[b]
            p = sigmoid(w @ d)
            g = (p - y)*d + l2*w
            w -= lr*g
    return w

def eval_metrics(pairs_sub, w):
    ys = pairs_sub.y_a_wins.to_numpy().astype(int)
    a_ids = pairs_sub.a_id.to_numpy().astype(int)
    b_ids = pairs_sub.b_id.to_numpy().astype(int)
    d = X[a_ids] - X[b_ids]
    ps = sigmoid(d @ w)
    acc = float(np.mean((ps >= 0.5) == (ys == 1)))
    eps=1e-12
    logloss = float(-np.mean(ys*np.log(ps+eps) + (1-ys)*np.log(1-ps+eps)))
    brier = float(np.mean((ps-ys)**2))
    return {"acc":acc, "logloss":logloss, "brier":brier}

train = pairs[pairs.split=="train"].reset_index(drop=True)
val   = pairs[pairs.split=="val"].reset_index(drop=True)
test  = pairs[pairs.split=="test"].reset_index(drop=True)

w_hat = train_pairwise_logreg(train)
metrics = pd.DataFrame([
    {"split":"train", **eval_metrics(train, w_hat)},
    {"split":"val", **eval_metrics(val, w_hat)},
    {"split":"test", **eval_metrics(test, w_hat)},
])
w_hat, metrics


## Как использовать в онлайне (идея)

- при каждом выборе пользователя сохраняем пару (A,B,choice)
- обновляем **w** одной SGD-итерацией (стриминг)
- генерируем новые кандидаты EQ (по перцептивным параметрам), ранжируем по \(w^T x\)
- лучший кандидат конвертируем в 23/11-полосный DSP профиль
